# Comparing Terraform state-management strategies

## Purpose

Terraform remembers what it built in a state file, and where that file lives decides how a workflow behaves once a second person (or a CI runner) shows up. This notebook compares three strategies side by side — the local backend on disk, a shared object-storage backend with a separate lock table, and Terraform Cloud remote execution — and works out which one fits solo work versus team work. This is one way to compare them; the docs for each backend also describe options not covered here.

## When to use each strategy

- **Local backend:** one person, throwaway or learning infrastructure, no CI writes. Zero setup.
- **Shared object storage plus lock table:** a small team sharing one state file who need locking and a durable copy outside any single laptop.
- **Terraform Cloud remote backend:** a team that also wants remote runs, per-workspace variables, and access control in one place instead of wiring storage, locking, and permissions separately.

## Prerequisites

- A Python interpreter with the standard library only (every code cell below runs on the standard library).
- Familiarity with the local-vs-remote state comparison in `tf/docs/local-vs-remote-terraform-state.md`.
- No cloud credentials needed: the cells render backend configuration text and exercise the decision logic locally, they do not contact any remote service.

In [ ]:
# last_verified: 2026-09-20 · terraform n/a
# Render the three backend configurations as text so their differences are visible.
# Nothing here is applied; the point is to compare shape and moving parts.

import os

WORKDIR = "/tmp/tf-state-compare"
os.makedirs(WORKDIR, exist_ok=True)

# Strategy 1: local backend. This is the default when no backend block exists,
# written out explicitly here so the three files can be diffed.
local_backend = '''terraform {
  backend "local" {
    path = "terraform.tfstate"
  }
}
'''

# Strategy 2: shared object storage for state plus a separate table for locking.
# Two resources to provision (bucket and lock table) plus access rules for who
# may read and write them.
s3_backend = '''terraform {
  backend "s3" {
    bucket         = "team-terraform-state"
    key            = "app/terraform.tfstate"
    region         = "us-east-1"
    dynamodb_table = "terraform-state-locks"
    encrypt        = true
  }
}
'''

# Strategy 3: Terraform Cloud remote backend. Storage, locking, and run history
# live in the hosted workspace; local runs only send configuration.
cloud_backend = '''terraform {
  backend "remote" {
    organization = "example-org"
    workspaces {
      name = "app-prod"
    }
  }
}
'''

files = {
    "backend-local.hcl": local_backend,
    "backend-s3.hcl": s3_backend,
    "backend-cloud.hcl": cloud_backend,
}
for name, body in files.items():
    with open(os.path.join(WORKDIR, name), "w") as fh:
        fh.write(body)
    print(f"wrote {name} ({len(body.splitlines())} lines)")
print("workdir:", WORKDIR)

## Steps

1. Run the rendering cell above and read the three files it writes. Note what each strategy asks you to provision: nothing, a bucket plus a lock table, or a hosted workspace.
2. Run the decision-helper cell below. It encodes the solo-vs-team rule of thumb as a small function and checks it against three scenarios.
3. Run the verification cell. It asserts that each rendered file contains the block the strategy depends on (local path, storage key plus lock table, hosted organization).
4. Read the decision table at the bottom and pick the strategy whose trade-offs match the team size being planned for.

In [ ]:
# Decision helper: which strategy fits a given way of working?
# Rule of thumb being encoded: solo throwaway work wants zero setup, a team
# sharing state needs locking plus durability, and a team that also wants
# hosted runs and access control points at the managed remote backend.

def recommend(collaborators, runs_in_ci, needs_hosted_runs):
    """Return the strategy name for a way of working."""
    if needs_hosted_runs:
        return "cloud"
    if collaborators == 1 and not runs_in_ci:
        return "local"
    return "s3"


scenarios = [
    # (collaborators, runs_in_ci, needs_hosted_runs, expected)
    (1, False, False, "local"),   # solo experiment on a laptop
    (3, True, False, "s3"),       # small team, CI applies, self-managed storage
    (6, True, True, "cloud"),     # larger team wanting hosted runs and access control
]
for collaborators, in_ci, hosted, expected in scenarios:
    got = recommend(collaborators, in_ci, hosted)
    assert got == expected, f"expected {expected}, got {got}"
    print(f"team={collaborators} ci={in_ci} hosted={hosted} -> {got} (as expected)")

In [ ]:
# Verify the rendered backend files contain the pieces each strategy depends on.
# Re-run this after any edit to the rendering cell; a failure means the
# comparison drifted from what the decision table below describes.

checks = {
    "backend-local.hcl": ['backend "local"', "terraform.tfstate"],
    "backend-s3.hcl": ['backend "s3"', "dynamodb_table", "encrypt"],
    "backend-cloud.hcl": ['backend "remote"', "organization", "workspaces"],
}
for name, needles in checks.items():
    with open(os.path.join(WORKDIR, name)) as fh:
        body = fh.read()
    for needle in needles:
        assert needle in body, f"{name} is missing {needle!r}"
    print(f"{name}: OK ({len(needles)} markers present)")
print("all backend files verified")

## Verify

Re-run every cell top to bottom in a fresh kernel. Expected result: three files appear under the printed workdir, the decision helper prints three `(as expected)` lines, and the verification cell ends with `all backend files verified`. If the verification cell fails, the rendering cell was edited without updating the decision table — fix the table, not the assertion.

## Decision table

| Situation | Preferred strategy | Reason |
|---|---|---|
| Solo, throwaway or learning infrastructure | Local backend | No setup; state on disk is fine while one person holds it |
| Small team sharing state, CI applies | Shared object storage plus lock table | Locking stops concurrent applies; state survives any single laptop |
| Team wanting hosted runs, variables, and access control in one place | Terraform Cloud remote backend | Storage, locking, and run history managed together per workspace |

The running theme: each step up adds one thing the previous level lacks — first locking and durability, then hosted execution and access control — at the cost of more to provision and permission.

## What I would try next

The natural follow-up is wiring the middle strategy for real: provisioning the storage bucket and lock table from the existing reusable module notes, then pointing a scratch configuration at them and watching a locked apply block a second one. That exercise would turn this comparison into a hands-on runbook.